# Laboratório Prático: Computação Paralela com OpenMP

**Disciplina:** Programação Concorrente
**Professor:** Gabriel P. Silva

**Duração:** ~2 horas

## Objetivos
- Empregar a diretiva `ordered` em laços `for`.
- Usar a cláusula `collapse` em laços aninhados.
- Analisar quando a barreira implícita pode ser removida com `nowait`.
- Comparar `critical` e `atomic`.
- Experimentar as cláusulas `atomic update`, `read`, `write` e `capture`.
---


## Exercício 1: Visualização do uso da diretiva ordered
A diretiva `#pragma omp ordered` permite que determinados trechos de código dentro do laço, protegidos pela clásula `ordered` sejam executados na mesma ordem que a execução sequencial do laço.

## Tarefas
1. Compile e execute o código. Observe o comportamento do programa.
2. Modifique o código comentando apenas a diretiva "ordered". Compile e execute o programa. Observe o comportamento.  
3. Modifique o código retirando apenas cláusula "ordered". Compile e execute o programa. Observe o comportamento.

Reporte as suas observações no formulário de respostas.

---



In [ ]:
%%writefile omp_ordered.c
#include <stdio.h>
#include <omp.h>

int main() {

    #pragma omp parallel for ordered schedule(static,1)
    for (int i = 0; i < 10; i++) {

        int tid = omp_get_thread_num();

        printf("[Thread %d] executou iteracao %d\n",
               tid, i);

        #pragma omp ordered
        {
            printf(">>> Saida ordenada: iteracao %d (thread %d)\n", i, tid);
        }
    }

    return 0;
}

Overwriting omp_ordered.c


In [ ]:
!gcc -fopenmp omp_ordered.c -o omp_ordered && ./omp_ordered

---
## Exercício 2: Collapse

A diretiva `collapse` permite que as iterações aninhadas de um laço sejam colapsadas para melhorar a distribuição de trabalho entre as threads disponíveis.

## Tarefa

1. Compile e execute o código a seguir como está. Observe o comportamento.
2. Adicione a cláusula `collapse (2)`. Compile e execute o código. Observe o comportamento.  
3. Em qual caso a divisão de trabalho tende a ser melhor? Reporte as suas observações no formulário de respostas.
---


In [ ]:
%%writefile omp_collapse.c
#include <stdio.h>
#include <omp.h>

int main() {
    int i, j;

    #pragma omp parallel for private(i,j) collapse(2) schedule(static)  num_threads(4)
    for (i = 0; i < 3; i++) {
        for (j = 0; j < 10; j++) {
            printf("tid=%d -> i=%d, j=%d\n", omp_get_thread_num(), i, j);
        }
    }
    return 0;
}

Overwriting omp_collapse.c


In [ ]:
!gcc -o omp_collapse -fopenmp omp_collapse.c
!./omp_collapse

tid=2 -> i=1, j=6
tid=2 -> i=1, j=7
tid=2 -> i=1, j=8
tid=2 -> i=1, j=9
tid=2 -> i=2, j=0
tid=2 -> i=2, j=1
tid=2 -> i=2, j=2
tid=0 -> i=0, j=0
tid=3 -> i=2, j=3
tid=3 -> i=2, j=4
tid=3 -> i=2, j=5
tid=3 -> i=2, j=6
tid=3 -> i=2, j=7
tid=3 -> i=2, j=8
tid=3 -> i=2, j=9
tid=1 -> i=0, j=8
tid=1 -> i=0, j=9
tid=1 -> i=1, j=0
tid=1 -> i=1, j=1
tid=1 -> i=1, j=2
tid=1 -> i=1, j=3
tid=1 -> i=1, j=4
tid=1 -> i=1, j=5
tid=0 -> i=0, j=1
tid=0 -> i=0, j=2
tid=0 -> i=0, j=3
tid=0 -> i=0, j=4
tid=0 -> i=0, j=5
tid=0 -> i=0, j=6
tid=0 -> i=0, j=7


---
## Exercício 3: Cláusula nowait e diretiva barrier
A cláusula nowait remove a barreira implícita no final das diretivas `for, sections e single`, mas isso deve ser feito com cuidado para não quebrar o programa.

##Atividades
1. Analise o código a seguir e modifique-o colocando barreiras explícitas com a diretiva `barrier` para garantir a execução correta do programa.  Coloque o seu código com as modificações e justificativas no formulário de respostas.
---

In [ ]:
%%writefile omp_nowait.c
#include <stdio.h>
#include <omp.h>
#include <stdlib.h>

int main() {
#define N 10000

int a[N], b[N], c[N], d[N], e[N];


    #pragma omp parallel num_threads(16) shared(a,b,c, d, e)
    {
          #pragma omp for schedule(dynamic,1) nowait // Laço 1
          for (int i = 0; i < N; i++){
              a[i] = c[i] + 1;
          }

          #pragma omp for schedule(dynamic,1)  nowait // Laço 2
          for(int i=0; i<N; i++)
               b[i] = i;

          #pragma omp for schedule(dynamic,1) nowait // Laço 3
          for (int i = 0; i < N; i++){
              c[i] = a[i] + 1;
          }

          #pragma omp for schedule(dynamic,1) nowait // Laço 4
          for(int i=0; i<N; i++)
               d[i] = d[i] + 2*i;

          #pragma omp for schedule(dynamic,1) nowait // Laço 5
          for (int i = 0; i < N; i++)
             e[i] = c[i] + 1;
      }

    return 0;
}

Overwriting omp_nowait_v2.c


In [ ]:
!gcc -fopenmp omp_nowait.c -o omp_nowait && ./omp_nowait

---
## Exercício 4: Uso de atomic

## Atividade

Complete cada trecho com uma das diretivas:

  `#pragma omp atomic read`

   `#pragma omp atomic write`

  `#pragma omp atomic update`

  `#pragma omp atomic capture`

Coloque o seu código com as modificações e justificativas no formulário de respostas.

---


In [ ]:
%%writefile omp_atomic.c
#include <stdio.h>
#include <omp.h>

int main() {

    int x = 100;
    int y = 50;
    int z;

    printf("Complete cada trecho com uma das diretivas:\n");
    printf("  #pragma omp atomic read\n");
    printf("  #pragma omp atomic write\n");
    printf("  #pragma omp atomic update\n");
    printf("  #pragma omp atomic capture\n\n");

    /* Trecho A */
    z = x;

    /* Trecho B */
    x = y;

    /* Trecho C */
    x++;

    /* Trecho D */
    x = x + y;

    /* Trecho E */
    x += y;

    /* Trecho F */
    x *= 2;

    /* Trecho G */
    z = x++;

    /* Trecho H */
    z = ++x;

    /* Trecho I */
    x = x - 1;

    /* Trecho J */
    {
        z = x;
        x++;
    }

    printf("x=%d y=%d z=%d\n", x, y, z);

    return 0;
}

---
## Exercício 5: Uso de critical e atomic

## Atividade

1. Executar várias vezes o programa a seguir e verificar se os resultados são consistentes.
2. Modificar o código e incluir a diretiva `critical` adequada de forma a maximizar o desempenho. Compilar, executar e verificar o funcionamento.  
3. Modificar o código e incluir a diretiva `atomic` com a clásula correta de forma a maximizar o desempenho. Compilar, executar e verificar o funcionamento.
4. Quais das duas diretivas você considera mais adequada para o problema em questão?

Coloque apenas o trecho de código com as modificações e justificativas no formulário de respostas.

---


In [ ]:
#include <stdio.h>
#include <omp.h>
#define ARESTAS 10
#define VERTICES 7

int main (int argc, char *argv[]) { /* omp_atomic.c  */
int i,j;
int num_arestas = ARESTAS, grau[VERTICES]={0,0,0,0,0,0,0};
typedef struct {
   int vertice1, vertice2;
   } tipo_aresta;

tipo_aresta aresta[ARESTAS];

    aresta[0].vertice1 = 0;
    aresta[0].vertice2 = 1;
    aresta[1].vertice1 = 0;
    aresta[1].vertice2 = 2;
    aresta[2].vertice1 = 1;
    aresta[2].vertice2 = 2;
    aresta[3].vertice1 = 1;
    aresta[3].vertice2 = 3;
    aresta[4].vertice1 = 2;
    aresta[4].vertice2 = 3;
    aresta[5].vertice1 = 2;
    aresta[5].vertice2 = 5;
    aresta[6].vertice1 = 2;
    aresta[6].vertice2 = 4;
    aresta[7].vertice1 = 3;
    aresta[7].vertice2 = 5;
    aresta[8].vertice1 = 3;
    aresta[8].vertice2 = 4;
    aresta[9].vertice1 = 4;
    aresta[9].vertice2 = 6;

    omp_set_num_threads(4);

    #pragma omp parallel for num_threads(16)
    for (j = 0; j< num_arestas; j++){
        grau[aresta[j].vertice1]++;
        grau[aresta[j].vertice2]++;
    }
    for (i = 0; i < VERTICES; i++)
   	    printf("Grau do vértice %d = %d \n",i, grau[i]);
   	return(0);
}